# FD-24 Сессия 3: Финализация — Стэкинг, Абляция и train_best.yaml (Слоты 8–9)
- **Слот 8:** Single-split стэкинг M1 + M2 на отложенных предсказаниях `val_real`
- **Слот 9:** E3 — абляция БЕЗ `diff_code` (`--drop_diff`) на лучшем бэкбоне
- **Финализация:** Автоматический выбор победителя по max PR-AUC на `val_real` и формирование `config/train_best.yaml`

In [ ]:
%cd /kaggle/working
import os
from kaggle_secrets import UserSecretsClient
try:
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo = f"https://{_tok}@github.com/o1uh/FreshDoc.git"
    print("clone: private (token)")
except Exception:
    _repo = "https://github.com/o1uh/FreshDoc.git"
    print("clone: public fallback (repo must be public)")

!if [ ! -d FreshDoc ]; then git clone --quiet {_repo} > /tmp/clone.log 2>&1 || (echo "CLONE FAILED:"; sed -E 's#(https?://)[^@]+@#\1***@#g' /tmp/clone.log); fi
if '_tok' in locals(): del _tok
if '_repo' in locals(): del _repo

%cd /kaggle/working/FreshDoc
!git checkout feat/FD-24-kaggle-experiments 2>/dev/null || !git checkout main 2>/dev/null || true
!git pull --ff-only || echo "PULL SKIPPED: убедись, что код свежий"

import os
os.environ["NCCL_P2P_DISABLE"] = "1"
!python -c "import transformers, sys; from packaging import version; sys.exit(0 if version.parse(transformers.__version__) >= version.parse('4.48.0') else 1)" || pip install --quiet --no-deps "transformers>=4.48.0"
!nvidia-smi

In [ ]:
import os, glob
import pandas as pd
_cands = glob.glob("/kaggle/input/**/dataset_v1.parquet", recursive=True)
assert _cands, "dataset_v1.parquet не найден в /kaggle/input (добавь датасет в Inputs блокнота)"
_p = _cands[0]
os.environ["DATA_PATH"] = _p
print(f"Dataset path: {_p}")

_df = pd.read_parquet(_p, columns=["split"])
assert len(_df) == 27203, f"dataset rows {len(_df)} != 27203"
_exp = {"train": 21997, "val": 559, "test_real": 2895, "test_synth": 1552, "gold": 200}
_got = _df["split"].value_counts().to_dict()
assert _got == _exp, f"split mismatch: {_got}"
print("dataset OK:", _got)
del _df

In [ ]:
# Генерация бейзлайнов M0 и M1 (подготовка для evaluate и стэкинга)
!python scripts/baselines_m0_m1.py --dataset "$DATA_PATH" \
  --out_m0 /kaggle/working/m0.csv --out_m1 /kaggle/working/m1.csv \
  --out_model /kaggle/working/m1_hgb.joblib --out_report /kaggle/working/m1_report.json

### Поиск чекпоинта лучшего бэкбона

In [ ]:
import os, glob, shutil, pandas as pd
# Копирование runs.csv из inputs в working при наличии, предотвращая PermissionError [Errno 30] и рассинхрон
local_runs = os.path.abspath('reports/runs.csv')
input_runs = sorted(glob.glob('/kaggle/input/**/runs.csv', recursive=True), key=os.path.getmtime, reverse=True)
if input_runs:
    print(f'Копирование актуального runs.csv из inputs: {input_runs[0]} -> {local_runs}')
    shutil.copy(input_runs[0], local_runs)
runs_path = local_runs

df_runs = pd.read_csv(runs_path)
is_neural = ~df_runs['model_name'].astype(str).str.startswith('stacking')
is_not_ablation = ~df_runs.get('drop_diff', pd.Series(False, index=df_runs.index)).astype(str).str.lower().isin(['true', '1'])
neural_runs = df_runs.loc[is_neural & is_not_ablation].dropna(subset=['eval_pr_auc'])
# Исключение legacy запусков, чьих чекпоинтов нет ни в working, ни в inputs
def _has_ckpt(d):
    name = os.path.basename(str(d))
    return os.path.exists(os.path.join(str(d), 'best')) or os.path.exists(os.path.join(str(d), 'preds.csv')) or \
           bool(glob.glob(f'/kaggle/input/**/{name}/best', recursive=True)) or \
           bool(glob.glob(f'/kaggle/input/**/{name}/preds.csv', recursive=True))
neural_runs = neural_runs.loc[neural_runs['output_dir'].apply(_has_ckpt)]
best_run = neural_runs.sort_values(by=['eval_pr_auc', 'timestamp'], ascending=[False, False]).iloc[0]

# Поиск чекпоинта и готовых предсказаний в working либо в подключенных inputs
best_ckpt = os.path.join(best_run['output_dir'], 'best')
best_preds = os.path.join(best_run['output_dir'], 'preds.csv')
ckpt_name = os.path.basename(best_run['output_dir'])
if not os.path.exists(best_ckpt):
    _cands = glob.glob(f'/kaggle/input/**/{ckpt_name}/best', recursive=True)
    if _cands:
        best_ckpt = _cands[0]
if not os.path.exists(best_preds):
    _cands_p = glob.glob(f'/kaggle/input/**/{ckpt_name}/preds.csv', recursive=True)
    if _cands_p:
        best_preds = _cands_p[0]
print(f"Победитель бэкбонов: {best_run['model_name']} (val PR-AUC: {best_run['eval_pr_auc']:.5f}) из {best_ckpt}")


In [ ]:
# ==============================================================================
# Слот 8: E2b — single-split стэкинг M1 + M2 на отложенных предсказаниях val_real
# ==============================================================================
# Использование готовых предсказаний M2 для исключения холостого повторного прогона 27k строк через GPU (~5-8 мин)
# Запуск стэкинга с передачей готовых предсказаний M1 и M2
if os.path.exists(best_preds):
    !python scripts/train_stacking.py --dataset "$DATA_PATH" \
      --m1_preds /kaggle/working/m1.csv \
      --m2_preds "{best_preds}" \
      --output_dir /kaggle/working/checkpoints/slot8_stacking \
      --out_preds /kaggle/working/checkpoints/slot8_stacking/m_stack.csv \
      --runs_csv "{runs_path}"
else:
    !python scripts/train_stacking.py --dataset "$DATA_PATH" \
      --m1_preds /kaggle/working/m1.csv \
      --m2_checkpoint "{best_ckpt}" --batch_size 16 \
      --output_dir /kaggle/working/checkpoints/slot8_stacking \
      --out_preds /kaggle/working/checkpoints/slot8_stacking/m_stack.csv \
      --runs_csv "{runs_path}"

# Оценка стэкинга через evaluate.py
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv \
  --pred M2=/kaggle/working/checkpoints/slot8_stacking/m_stack.csv \
  --out_metrics /kaggle/working/checkpoints/slot8_stacking/metrics.json \
  --out_table /kaggle/working/checkpoints/slot8_stacking/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot8_stacking/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot8_stacking/table_e1.csv


In [ ]:
# ==============================================================================
# Слот 9: E3 — Абляция БЕЗ diff_code (--drop_diff) на архитектуре лучшего бэкбона
# ==============================================================================
# Определение лучшего бэкбона, если Cell 5 не запускалась в текущей сессии
if 'best_run' not in locals():
    import os, glob, pandas as pd
    df_runs = pd.read_csv(runs_path if 'runs_path' in locals() else 'reports/runs.csv')
    is_neural = ~df_runs['model_name'].astype(str).str.startswith('stacking')
    is_not_ablation = ~df_runs.get('drop_diff', pd.Series(False, index=df_runs.index)).astype(str).str.lower().isin(['true', '1'])
    neural_runs = df_runs.loc[is_neural & is_not_ablation].dropna(subset=['eval_pr_auc'])
    def _has_ckpt(d):
        name = os.path.basename(str(d))
        return os.path.exists(os.path.join(str(d), 'best')) or os.path.exists(os.path.join(str(d), 'preds.csv')) or \
               bool(glob.glob(f'/kaggle/input/**/{name}/best', recursive=True)) or \
               bool(glob.glob(f'/kaggle/input/**/{name}/preds.csv', recursive=True))
    neural_runs = neural_runs.loc[neural_runs['output_dir'].apply(_has_ckpt)]
    best_run = neural_runs.sort_values(by=['eval_pr_auc', 'timestamp'], ascending=[False, False]).iloc[0]

best_model_name = best_run["model_name"]
best_max_len = int(best_run["max_len"])
gc_flag = "--gradient_checkpointing" if best_max_len >= 2048 else ""

!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name "{best_model_name}" \
  --max_len {best_max_len} {gc_flag} --drop_diff --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot9_ablation_no_diff --runs_csv "{runs_path}"
!rm -rf /kaggle/working/checkpoints/slot9_ablation_no_diff/checkpoint-*

# Инференс с флагом --drop_diff
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot9_ablation_no_diff/best \
  --dataset "$DATA_PATH" --batch_size 16 --drop_diff --out /kaggle/working/checkpoints/slot9_ablation_no_diff/preds.csv

# Оценка абляции
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv \
  --pred M2=/kaggle/working/checkpoints/slot9_ablation_no_diff/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot9_ablation_no_diff/metrics.json \
  --out_table /kaggle/working/checkpoints/slot9_ablation_no_diff/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot9_ablation_no_diff/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot9_ablation_no_diff/table_e1.csv


In [ ]:
# ==============================================================================
# Финализация: автоматический выбор лучшего бэкбона по val_real PR-AUC
# ==============================================================================
!python scripts/finalize_best_config.py --runs_csv "{runs_path}" \
  --out_config config/train_best.yaml
!cat config/train_best.yaml

import pandas as pd
df_runs = pd.read_csv(runs_path)
print('\n=== ИТОГОВАЯ ТАБЛИЦА ВСЕХ ПРОГОНОВ FD-24 ===')
print(df_runs.to_string(index=False))
